In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Lab4").getOrCreate()

## 1. Hãy đọc dữ liệu từ các file csv, sử dụng tự suy ra kiểu dữ liệu cho mỗi cột.

In [4]:
# load data
df_orders = spark.read.csv("Orders.csv", header=True, sep=";", inferSchema=True)
df_customers = spark.read.csv("Customer_List.csv", header=True, sep=";", inferSchema=True)
df_products = spark.read.csv("Products.csv", header=True, sep=";", inferSchema=True)
df_order_items = spark.read.csv("Order_Items.csv", header=True, sep=";", inferSchema=True)
df_order_reviews = spark.read.csv("Order_Reviews.csv", header=True, sep=";", inferSchema=True)

## 2. Thống kê tổng số đơn hàng, số lượng khách hàng và người bán.

In [5]:
# count basic stats
count_orders = df_orders.count()
print("Total orders: ", count_orders)

count_customers = df_customers.select("Customer_Trx_ID").distinct().count()
print("Total customers: ", count_customers)

count_sellers = df_order_items.select("seller_id").distinct().count()
print("Total selers: ", count_sellers)

Total orders:  99441
Total customers:  99442
Total selers:  3095


## 3. Phân tích số lượng đơn hàng theo quốc gia, sắp xếp theo thứ tự giảm dần.

In [7]:
from pyspark.sql.functions import desc

# orders by country
country_analysis = df_orders.join(df_customers, "Customer_Trx_ID") \
    .groupBy("Customer_Country") \
    .count() \
    .orderBy(desc("count"))

country_analysis.show(20)

+----------------+-----+
|Customer_Country|count|
+----------------+-----+
|         Germany|41754|
|          France|12848|
|     Netherlands|11629|
|         Belgium| 5464|
|         Austria| 5043|
|     Switzerland| 3640|
|  United Kingdom| 3382|
|          Poland| 2139|
|         Czechia| 2034|
|           Italy| 2025|
|           Spain| 1651|
|        Portugal| 1336|
|          Sweden|  975|
|         Denmark|  905|
|          Serbia|  746|
|          Norway|  716|
|        Slovakia|  534|
|        Slovenia|  495|
|          Turkey|  485|
|          Greece|  412|
+----------------+-----+
only showing top 20 rows


## 4. Phân tích số lượng đơn hàng nhóm theo năm, tháng đặt hàng (Hiển thị theo năm tăng dần, tháng giảm dần)

In [8]:
from pyspark.sql.functions import to_timestamp, year, month, asc

# orders by year and month
df_time = df_orders.withColumn("timestamp", to_timestamp("Order_Purchase_Timestamp")) \
    .withColumn("year", year("timestamp")) \
    .withColumn("month", month("timestamp")) \
    .dropna(subset=["year", "month"])

time_analysis = df_time.groupBy("year", "month") \
    .count() \
    .orderBy(asc("year"), desc("month"))

time_analysis.show(20)

+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2022|   12|    1|
|2022|   10|  324|
|2022|    9|    4|
|2023|   12| 5673|
|2023|   11| 7544|
|2023|   10| 4631|
|2023|    9| 4285|
|2023|    8| 4331|
|2023|    7| 4026|
|2023|    6| 3245|
|2023|    5| 3700|
|2023|    4| 2404|
|2023|    3| 2682|
|2023|    2| 1780|
|2023|    1|  800|
|2024|   10|    4|
|2024|    9|   16|
|2024|    8| 6512|
|2024|    7| 6292|
|2024|    6| 6167|
+----+-----+-----+
only showing top 20 rows


## 5. Thống kê điểm đánh giá trung bình, số lượng đánh giá theo từng mức (ví dụ: 1 đến 5).


In [20]:
from pyspark.sql.functions import col, avg

# Q5: filter valid scores (1-5) and cast to int
df_reviews = df_order_reviews.filter(col("Review_Score").isin(["1", "2", "3", "4", "5"]))
df_reviews = df_reviews.withColumn("Review_Score", col("Review_Score").cast("int"))

# calculate average review score
avg_review = df_reviews.select(avg("Review_Score")).first()[0]
print("Avg point:", avg_review)

# count reviews by score
count_review = df_reviews.groupBy("Review_Score").count().orderBy("Review_Score")
count_review.show()

Avg point: 4.0864214950162765
+------------+-----+
|Review_Score|count|
+------------+-----+
|           1|11424|
|           2| 3151|
|           3| 8179|
|           4|19141|
|           5|57328|
+------------+-----+



## 6. Tính doanh thu (giá sản phẩm + phí vận chuyển) trong năm 2024 và nhóm theo danh mục sản phẩm

In [26]:
from pyspark.sql.functions import col, sum

# filter orders in 2024
df_orders_2024 = df_time.filter(col("year") == 2024).select("Order_ID")

# join orders 2024 with order_items
df_order_items_2024 = df_order_items.join(
    df_orders_2024,
    on="Order_ID",
    how="inner"
)

# join with products
df_product_items_2024 = df_order_items_2024.join(
    df_products.select("Product_ID", "Product_Category_Name"),
    on="Product_ID",
    how="inner"
)

# calculate total value (price + freight value) for each item
df_revenue_items = df_product_items_2024.withColumn(
    "Total_Value",
    col("Price").cast("float") + col("Freight_Value").cast("float")
)

# group by category and sum revenue
category_revenue = df_revenue_items.groupBy("Product_Category_Name") \
    .agg(
        sum("Total_Value").alias("Total_Revenue")
    ) \
    .orderBy("Total_Revenue", ascending=False)

# result
category_revenue.show(10, truncate=False)

+---------------------+------------------+
|Product_Category_Name|Total_Revenue     |
+---------------------+------------------+
|Health_Beauty        |885191.1222515106 |
|Watches_Gifts        |771986.7499084473 |
|Bed_Bath_Table       |650794.6994714737 |
|Sports_Leisure       |621999.3395881653 |
|Computers_Accessories|594771.0368652344 |
|Housewares           |491576.96055984497|
|Furniture_Decor      |476466.13181209564|
|Auto                 |404210.57095718384|
|Baby                 |299052.5591850281 |
|Cool_Stuff           |273910.0507850647 |
+---------------------+------------------+
only showing top 10 rows


## 7. Xác định sản phẩm có số lượng bán ra cao nhất và tính điểm đánh giá trung bình cho từng sản phẩm

In [29]:
from pyspark.sql.functions import count, avg

# join data
df_order_join_items = df_order_items.join(
    df_orders.select("Order_ID"),
    on="Order_ID",
    how="inner"
)

df_product_join_order_items = df_order_join_items.join(
    df_products.select("Product_ID", "Product_Category_Name"),
    on="Product_ID",
    how="inner"
)

df_product_item_reviews = df_product_join_order_items.join(
    df_reviews.select("Order_ID", "Review_Score"),
    on="Order_ID",
    how="inner"
)


# group by product to calculate total sold and average rating
product_stats = df_product_item_reviews.groupBy("Product_ID", "Product_Category_Name") \
    .agg(
        count("*").alias("Total_Sold"),
        avg("Review_Score").alias("Avg_Rating")
    ) \
    .orderBy("Total_Sold", ascending=False)


# print top 1 best-selling product
top_product = product_stats.first()
print(f"Top 1 best-selling product is ID {top_product['Product_ID']} with {top_product['Total_Sold']} orders.\n")

# show the final table
product_stats.show(10, truncate=False)

Top 1 best-selling product is ID aca2eb7d00ea1a7b8ebd4e68314663af with 524 orders.

+--------------------------------+---------------------+----------+------------------+
|Product_ID                      |Product_Category_Name|Total_Sold|Avg_Rating        |
+--------------------------------+---------------------+----------+------------------+
|aca2eb7d00ea1a7b8ebd4e68314663af|Furniture_Decor      |524       |4.019083969465649 |
|422879e10f46682990de24d770e7f83d|Garden_Tools         |486       |3.9465020576131686|
|99a4788cb24856965c36a24e339b6058|Bed_Bath_Table       |482       |3.8983402489626555|
|389d119b48cf3043d311335e499d9c6b|Garden_Tools         |391       |4.117647058823529 |
|368c6c730842d78016ad823897a372db|Garden_Tools         |388       |3.922680412371134 |
|53759a2ecddad2bb87a079a1f1519f73|Garden_Tools         |373       |3.868632707774799 |
|d1c427060a0f73f6b889a5c7c61f2ac4|Computers_Accessories|340       |4.194117647058824 |
|53b36df67ebb7c41585e8d54d6772e08|Watches_Gift